Import packages 

In [7]:
import duckdb
import pyarrow.parquet as pq
import os
import re
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# RAG import packages 
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# Vector Embeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# import from directory
from llm import llm

load_dotenv("../.env")  # .env is in data_concierge (parent of retrieval)

DATASET_DIR = "dataset"  # or "retrieval/dataset" if running from project root
CSV_OUTPUT_DIR = "csv_files"
KEY_NAME = os.getenv("PARQUET_KEY_NAME")
KEY_VALUE = os.getenv("PARQUET_KEY_VALUE")

Exporting all parquet file as a csv file

In [8]:
if False:
    os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

    con = duckdb.connect()
    con.execute(f"PRAGMA add_parquet_key('{KEY_NAME}', '{KEY_VALUE}');")

    for f in os.listdir(DATASET_DIR):
        if f.endswith(".parquet"):
            parquet_path = os.path.join(DATASET_DIR, f)
            csv_path = os.path.join(CSV_OUTPUT_DIR, f.replace(".parquet", ".csv"))
            con.execute(f"""
                COPY (
                    SELECT * FROM read_parquet(
                        '{parquet_path}',
                        encryption_config = {{footer_key: '{KEY_NAME}'}}
                    )
                ) TO '{csv_path}' (HEADER, DELIMITER ',');
            """)
            print(f"Exported {f} -> {csv_path}")


Parse txt files into column-level chunks

In [9]:
def parse_schema_to_column_chunks(docs_dir: str) -> list[Document]:
    """Parse each COLUMN block into a separate Document with table metadata."""
    chunks = []
    
    for filename in os.listdir(docs_dir):
        if not filename.endswith(".txt"):
            continue
            
        filepath = os.path.join(docs_dir, filename)
        table_name = filename.replace(".txt", "")
        
        with open(filepath, "r") as f:
            content = f.read()
        
        # Extract TABLE line for context
        table_match = re.search(r"TABLE: (\w+) \((\d+) rows\)", content)
        joins_match = re.search(r"JOINS: (.+)", content)
        joins = joins_match.group(1) if joins_match else ""
        
        # Split by COLUMN blocks
        column_blocks = re.split(r"\n(?=COLUMN(?: NAME)?: )", content)
        
        for block in column_blocks:
            if not block.strip() or block.startswith("TABLE:") or block.startswith("JOINS:"):
                continue
            col_match = re.search(r"COLUMN NAME: (.+?) \(Use this when querying information\)", block)
            if not col_match:
                col_match = re.search(r"COLUMN: (\w+(?:\s+\w+)?)", block)
            if not col_match:
                continue
            col_name = col_match.group(1).strip()
            
            # Full column block as chunk content (Data Element, Description, etc.)
            chunk_content = f"TABLE: {table_name}\n{block.strip()}"
            if joins:
                chunk_content += f"\n\nJOINS: {joins}"
            
            doc = Document(
                page_content=chunk_content,
                metadata={
                    "table": table_name,
                    "column": col_name,
                    "source": filename
                }
            )
            chunks.append(doc)
    
    return chunks

Load into vector store and run retrieval

In [10]:
# Parse to column-level chunks
docs_dir = "documents_for_vectordb"
chunks = parse_schema_to_column_chunks(docs_dir)

load_dotenv("../.env")

# OpenRouter Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Then use embeddings in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="schema_columns",
    persist_directory="./chroma_db"
)

# Retrieve relevant columns for a user query
query = "How many patients died and what was the cause?"
retrieved = vectorstore.similarity_search(query, k=5)

for doc in retrieved:
    print(f"Table: {doc.metadata['table']}, Column: {doc.metadata['column']}")
    print(doc.page_content[:200], "...\n")

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column

RAG chain for NL-to-SQL

In [11]:
# Prompt for SQL generation
SQL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a SQL expert. Generate DuckDB SQL queries based on the schema context provided.
Use only the tables and columns mentioned in the context. Table names and column names are case-sensitive.
If joining tables, use person_id as the common key."""),
    ("human", """Schema context:
{context}

User question: {question}

Generate the SQL query only. No explanation.""")
])

# RAG chain
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": lambda x: format_docs(vectorstore.similarity_search(x["question"], k=6)),
     "question": lambda x: x["question"]}
    | SQL_PROMPT
    | llm
    | StrOutputParser()
)

# Use it
sql = rag_chain.invoke({"question": "Count patients by gender"})
print(sql)

SELECT gender_concept_id, COUNT(DISTINCT person_id) AS patient_count
FROM person
GROUP BY gender_concept_id
ORDER BY gender_concept_id;


Parse cancer.txt and sql_template.txt, create vector databases

Build Chroma collections:
- **icd10_cancer_reference**: ICD-10-AM cancer codes and descriptions from cancer.txt (for mapping cancer types to codes)
- **sql_templates**: Few-shot SQL examples from sql_template.txt (for retrieving relevant query patterns)

In [12]:
# Paths (relative to retrieval/)
CANCER_PATH = "icd-am/cancer.txt"
SQL_TEMPLATE_PATH = "sql_template/sql_template.txt"
CHROMA_DIR = "./chroma_db"

In [13]:
def parse_cancer_txt(path: str) -> list[Document]:
    """Parse cancer.txt into Documents by section. Sections are separated by ==== lines; title and content alternate."""
    with open(path, "r") as f:
        content = f.read()
    parts = re.split(r"\n=+\n", content)
    docs = []
    # parts[0] = intro; parts[1]=title, parts[2]=content, parts[3]=title, parts[4]=content, ...
    for i in range(1, len(parts) - 1, 2):
        title = parts[i].strip()
        content_block = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if not title or not content_block:
            continue
        combined = f"{title}\n\n{content_block}"
        doc = Document(
            page_content=combined,
            metadata={"source": "cancer.txt", "type": "icd10_cancer_reference"},
        )
        docs.append(doc)
    return docs

In [14]:
# Parse and build cancer vector DB
cancer_docs = parse_cancer_txt(CANCER_PATH)
print(f"Parsed {len(cancer_docs)} cancer reference sections")

cancer_vectorstore = Chroma.from_documents(
    documents=cancer_docs,
    embedding=embeddings,
    collection_name="icd10_cancer_reference",
    persist_directory=CHROMA_DIR,
)
print("Created chroma collection: icd10_cancer_reference")


Parsed 12 cancer reference sections
Created chroma collection: icd10_cancer_reference


Testing the cancer db

In [15]:
print("\nTest: cancer query 'colorectal' ->", cancer_vectorstore.similarity_search("colorectal cancer", k=1)[0].page_content[:150], "...")


Test: cancer query 'colorectal' -> COLORECTAL CANCER (C18–C21)

C18  Colon cancer
  C18.0  Malignant neoplasm of caecum
  C18.1  Malignant neoplasm of appendix
  C18.2  Malignant neopla ...


Parsing sql template documents

In [16]:
def parse_sql_template_txt(path: str) -> list[Document]:
    """Parse sql_template.txt into Documents by block (split by ---)."""
    with open(path, "r") as f:
        content = f.read()
    # Skip header comment lines
    blocks = content.split("\n---\n")
    docs = []
    for block in blocks:
        block = block.strip()
        if not block or block.startswith("#"):
            continue
        doc = Document(
            page_content=block,
            metadata={"source": "sql_template.txt", "type": "sql_template"},
        )
        docs.append(doc)
    return docs

In [17]:
# Parse and build SQL template vector DB
sql_docs = parse_sql_template_txt(SQL_TEMPLATE_PATH)
print(f"Parsed {len(sql_docs)} SQL template blocks")

sql_template_vectorstore = Chroma.from_documents(
    documents=sql_docs,
    embedding=embeddings,
    collection_name="sql_templates",
    persist_directory=CHROMA_DIR,
)
print("Created chroma collection: sql_templates")



Parsed 14 SQL template blocks
Created chroma collection: sql_templates


Test for the sql template db

In [18]:
print("\nTest: sql query 'patients 2020' ->", sql_template_vectorstore.similarity_search("total patients diagnosed 2020", k=1)[0].page_content[:200], "...")


Test: sql query 'patients 2020' -> HEADER: What was the total number of patients diagnosed with colorectal cancer in 2020?
TIP: Use COUNT(DISTINCT person_id) because a patient might have 2 colorectal cancer diagnoses in same year — do  ...
